# User-to-Item Recommendation System with BERT
Unlike bag-of-words and TF-IDF, BERT produces dense semantic embeddings that can capture similarity even when wines are described with different words.

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
wines_features = pd.read_csv(
    Path("../data/processed/wines_features.csv"),
    usecols=["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity", "bert_text"]
)

ratings = pd.read_csv(
    Path("../dataset/XWines_Full_21M_ratings.csv"),
    usecols=["UserID", "WineID", "Rating"],
    dtype={"UserID": "int32", "WineID": "int32", "Rating": "float32"}
)

wines_features = wines_features.drop_duplicates(subset="WineID").copy()
wines_features = wines_features[wines_features["bert_text"].notna()].copy()
wines_features = wines_features[wines_features["bert_text"].str.strip() != ""].copy()

wines_features["WineID"] = wines_features["WineID"].astype("int32")
ratings["WineID"] = ratings["WineID"].astype("int32")
ratings = ratings.drop_duplicates(subset=["UserID", "WineID"]).copy()

print("wines_features:", wines_features.shape)
print("ratings:", ratings.shape)

wines_features: (100646, 8)
ratings: (20590800, 3)


In [ ]:
POSITIVE_THRESHOLD = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 200
MAX_USERS = 5000
RANDOM_STATE = 42
TEST_FRACTION = 0.2
SAMPLED_USERS_PATH = Path("../data/results/user_to_item_sampled_users_5000.csv")

In [ ]:
positive = ratings[ratings["Rating"] >= POSITIVE_THRESHOLD].copy()
user_counts = positive["UserID"].value_counts()
eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index
positive = positive[positive["UserID"].isin(eligible_users)].copy()

if SAMPLED_USERS_PATH.exists():
    sampled_users = pd.read_csv(SAMPLED_USERS_PATH)["UserID"].to_numpy()
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_users = rng.choice(
        positive["UserID"].unique(),
        size=min(MAX_USERS, positive["UserID"].nunique()),
        replace=False
    )
    SAMPLED_USERS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(sampled_users).to_csv(SAMPLED_USERS_PATH, index=False, header=["UserID"])

positive = positive[positive["UserID"].isin(sampled_users)].copy()

print("users used in this run:", positive["UserID"].nunique())
print("positive interactions used:", positive.shape)

users used in this run: 5000
positive interactions used: (71243, 3)


In [ ]:
train_parts = []
test_parts = []

for user_id, group in positive.groupby("UserID"):
    group = group.sample(frac=1, random_state=RANDOM_STATE)
    n_test = max(1, int(np.ceil(len(group) * TEST_FRACTION)))

    test_group = group.iloc[:n_test]
    train_group = group.iloc[n_test:]

    if len(train_group) == 0:
        continue

    train_parts.append(train_group)
    test_parts.append(test_group)

train_pos = pd.concat(train_parts, ignore_index=True)
test_pos = pd.concat(test_parts, ignore_index=True)

print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)

train_pos: (55019, 3)
test_pos: (16224, 3)


In [ ]:
wine_texts = wines_features["bert_text"].tolist()

wine_embeddings = model.encode(
    wine_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(wine_embeddings.shape)

Batches:   0%|          | 0/1573 [00:00<?, ?it/s]

(100646, 384)


Instead of representing wines as sparse word counts we used BERT-based text embeddings which capture broader semantic similarity between wines.

In [ ]:
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()
idx_to_wineid = pd.Series(wines_features["WineID"].values, index=wines_features.index)

train_pos = train_pos[train_pos["WineID"].isin(wineid_to_idx.index)].copy()
test_pos = test_pos[test_pos["WineID"].isin(wineid_to_idx.index)].copy()

print("train_pos after filtering:", train_pos.shape)
print("test_pos after filtering:", test_pos.shape)

train_pos after filtering: (55019, 3)
test_pos after filtering: (16224, 3)


In [ ]:
def build_user_profile_bert(user_history):
    user_history = user_history[user_history["WineID"].isin(wineid_to_idx.index)].copy()

    item_indices = [wineid_to_idx[wine_id] for wine_id in user_history["WineID"]]
    weights = (user_history["Rating"] - 3.0).clip(lower=1.0).to_numpy()

    item_vectors = wine_embeddings[item_indices]
    profile = np.average(item_vectors, axis=0, weights=weights)

    norm = np.linalg.norm(profile)
    if norm > 0:
        profile = profile / norm

    return profile

In [ ]:
user_profiles_bert = {}
user_groups = list(train_pos.groupby("UserID"))

for i, (user_id, user_history) in enumerate(user_groups, start=1):
    user_profiles_bert[user_id] = build_user_profile_bert(user_history)

    if i % 500 == 0:
        print(f"Built {i}/{len(user_groups)} BERT user profiles")

print("built profiles:", len(user_profiles_bert))

Built 500/5000 BERT user profiles
Built 1000/5000 BERT user profiles
Built 1500/5000 BERT user profiles
Built 2000/5000 BERT user profiles
Built 2500/5000 BERT user profiles
Built 3000/5000 BERT user profiles
Built 3500/5000 BERT user profiles
Built 4000/5000 BERT user profiles
Built 4500/5000 BERT user profiles
Built 5000/5000 BERT user profiles
built profiles: 5000


In [ ]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles_bert.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

train_seen_idx = {
    user_id: [wineid_to_idx[w] for w in seen if w in wineid_to_idx]
    for user_id, seen in train_seen.items()
}

evaluation users: 5000


In [ ]:
def recommend_user_bert_ids(user_id, top_k=20, candidate_pool=400):
    if user_id not in user_profiles_bert:
        return []

    profile = user_profiles_bert[user_id].reshape(1, -1)
    scores = cosine_similarity(profile, wine_embeddings).ravel()

    seen_idx = train_seen_idx.get(user_id, [])
    if seen_idx:
        scores[seen_idx] = -np.inf

    pool = min(candidate_pool, scores.size)
    candidate_idx = np.argpartition(scores, -pool)[-pool:]
    candidate_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]

    return [int(idx_to_wineid[i]) for i in candidate_idx[:top_k]]

To make the model more efficient it first keeps only a limited pool of the highest scoring candidates instead of ranking the entire wine catalog.

In [ ]:
def show_user_recommendations_bert(user_id, top_k=10):
    rec_ids = recommend_user_bert_ids(user_id, top_k=top_k)

    recs = wines_features[wines_features["WineID"].isin(rec_ids)][
        ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
    ].copy()

    recs["rank"] = recs["WineID"].map({wine_id: i + 1 for i, wine_id in enumerate(rec_ids)})
    recs = recs.sort_values("rank").reset_index(drop=True)

    return recs

In [ ]:
sample_user = eval_users[0]
show_user_recommendations_bert(sample_user, top_k=10)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,180745,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,1
1,182516,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,2
2,180775,Pinot Noir,Red,United States,Willamette Valley,Medium-bodied,High,3
3,184995,Pinot Noir,Red,United States,Sta. Rita Hills,Medium-bodied,High,4
4,190648,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,5
5,190526,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,6
6,183191,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,7
7,180213,Pinot Noir,Red,United States,Central Coast,Medium-bodied,High,8
8,185144,S Pinot Noir,Red,United States,Willamette Valley,Medium-bodied,High,9
9,179277,Pinot Noir,Red,United States,Willamette Valley,Medium-bodied,High,10


In [ ]:
def precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k

def recall_at_k(relevant, recommended, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)

def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score

def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg

def average_precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    hits = 0
    score = 0.0

    for i, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / i

    denom = min(len(relevant), k)
    return score / denom if denom > 0 else 0.0

def hit_rate_at_k(relevant, recommended, k):
    return float(any(item in relevant for item in recommended[:k]))

def reciprocal_rank_at_k(relevant, recommended, k):
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [ ]:
def evaluate_recommender(recommend_func, users, test_relevant_dict, ks=(5, 10, 20)):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_dict.get(user_id, set())
        if not relevant:
            continue

        recommended = recommend_func(user_id, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [ ]:
bert_eval = evaluate_recommender(
    recommend_func=recommend_user_bert_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

bert_summary = (
    bert_eval
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="BERT")
)

bert_summary

Evaluated 500/5000 users
Evaluated 1000/5000 users
Evaluated 1500/5000 users
Evaluated 2000/5000 users
Evaluated 2500/5000 users
Evaluated 3000/5000 users
Evaluated 3500/5000 users
Evaluated 4000/5000 users
Evaluated 4500/5000 users
Evaluated 5000/5000 users


,BERT
Precision@5,0.0008
Recall@5,0.0015
NDCG@5,0.0011
MAP@5,0.0006
HitRate@5,0.0038
MRR@5,0.0016
Precision@10,0.0008
Recall@10,0.0031
NDCG@10,0.0018
MAP@10,0.0008


In [ ]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
bert_summary.to_csv("../data/results/bert_summary.csv")

In [ ]:
wines_features["eval_signature"] = (
    wines_features["Type"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Country"].astype(str).str.lower().str.strip() + "|" +
    wines_features["RegionName"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Body"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Acidity"].astype(str).str.lower().str.strip()
)

wineid_to_signature = dict(zip(wines_features["WineID"], wines_features["eval_signature"]))

test_relevant_signatures = {
    user_id: {wineid_to_signature[w] for w in wine_ids if w in wineid_to_signature}
    for user_id, wine_ids in test_relevant.items()
}

In [ ]:
def recommended_ids_to_signatures(recommended_ids, top_k=None):
    signatures = []
    seen_signatures = set()

    for wine_id in recommended_ids:
        if wine_id not in wineid_to_signature:
            continue

        signature = wineid_to_signature[wine_id]
        if signature in seen_signatures:
            continue

        signatures.append(signature)
        seen_signatures.add(signature)

        if top_k is not None and len(signatures) == top_k:
            break

    return signatures

In [ ]:
def evaluate_recommender_soft(recommend_func, users, test_relevant_signatures_dict, ks=(5, 10, 20), oversample_factor=5):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_signatures_dict.get(user_id, set())
        if not relevant:
            continue

        recommended_ids = recommend_func(user_id, top_k=max_k * oversample_factor)
        recommended = recommended_ids_to_signatures(recommended_ids, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Soft-evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [ ]:
bert_eval_soft = evaluate_recommender_soft(
    recommend_func=recommend_user_bert_ids,
    users=eval_users,
    test_relevant_signatures_dict=test_relevant_signatures,
    ks=(5, 10, 20),
    oversample_factor=5
)

bert_summary_soft = (
    bert_eval_soft
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="BERT Soft")
)

bert_summary_soft

Soft-evaluated 500/5000 users
Soft-evaluated 1000/5000 users
Soft-evaluated 1500/5000 users
Soft-evaluated 2000/5000 users
Soft-evaluated 2500/5000 users
Soft-evaluated 3000/5000 users
Soft-evaluated 3500/5000 users
Soft-evaluated 4000/5000 users
Soft-evaluated 4500/5000 users
Soft-evaluated 5000/5000 users


,BERT Soft
Precision@5,0.0586
Recall@5,0.1132
NDCG@5,0.1099
MAP@5,0.0790
HitRate@5,0.2536
MRR@5,0.1687
Precision@10,0.0385
Recall@10,0.1458
NDCG@10,0.1210
MAP@10,0.0820


In [ ]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
bert_eval_soft.to_csv("../data/results/bert_eval_soft.csv", index=False)
bert_summary_soft.to_csv("../data/results/bert_summary_soft.csv")

# Save the model

In [ ]:
from pathlib import Path
import pandas as pd

ARMS_DIR = Path("../bandits/saved_arms")
ARMS_DIR.mkdir(parents=True, exist_ok=True)

def export_arm_recs(recommend_fn, users, out_csv, top_k=100):
    rows = []
    for uid in users:
        recs = recommend_fn(int(uid), top_k=top_k)
        for r, wid in enumerate(recs, start=1):
            rows.append({"UserID": int(uid), "rank": int(r), "WineID": int(wid)})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)
    print(f"Saved {len(out_df):,} rows -> {out_csv}")


In [ ]:
export_arm_recs(
    recommend_fn=recommend_user_bert_ids,
    users=eval_users,
    out_csv=ARMS_DIR / "content_bert_recs.csv",
    top_k=100
)
